<a href="https://colab.research.google.com/github/heidy0099/Proy.Org/blob/main/CodigoFinalPloty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [180]:
from google.colab import files
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

print("SUBE TU ARCHIVO CSV:")
uploaded = files.upload()
archivo = list(uploaded.keys())[0]

df = pd.read_csv(archivo, sep=';', encoding='latin1')
df.columns = df.columns.str.strip()
df['Grupo'] = df.iloc[:,0].str.extract(r'([A-Z]+)')[0]

print("OK", df.shape[0], "reacciones")

SUBE TU ARCHIVO CSV:


Saving Proyecto 3 Quimica Organica TRATAMEINTO FINAL.csv to Proyecto 3 Quimica Organica TRATAMEINTO FINAL (39).csv
OK 242 reacciones


In [181]:
# GRAFICO 1
fig = px.scatter(df, x='PMS', y='PES', color='Grupo',
                 title='Grafico 1: Fuerzas de Dispersion de London')
fig.show()

In [182]:
# GRAFICO 2
fig = px.box(df, x='Grupo', y='PES', color='Grupo',
             title='Grafico 2: Fuerzas Intermoleculares por Familia')
fig.show()

In [183]:
# GRAFICO 3
pka_mean = df.groupby('Grupo')['pKa S'].mean().reset_index()
fig = px.bar(pka_mean, x='Grupo', y='pKa S', color='Grupo',
             title='Grafico 3: Acidez y Estabilidad por Resonancia',
             text='pKa S')
fig.update_traces(textposition='outside')
fig.show()

In [184]:
# GRAFICO 4
ld50_mean = df.groupby('Grupo')['LD5 S'].mean().reset_index()
fig = px.bar(ld50_mean, x='Grupo', y='LD5 S', color='Grupo',
             title='Grafico 4: Evaluacion de Riesgo Ambiental y Toxicidad',
             text='LD5 S')
fig.update_traces(textposition='outside')
fig.show()

In [185]:
# GRAFICO 5
conteo = df['Grupo'].value_counts().reset_index()
conteo.columns = ['Grupo', 'Cantidad']
fig = px.pie(conteo, values='Cantidad', names='Grupo', hole=0.3,
             title='Grafico 5: Distribucion de Reacciones por Grupo Funcional')
fig.show()

In [186]:
# GRAFICO 6
pm_reaccion = df.groupby('TR')['PMP'].mean().reset_index().dropna().sort_values('PMP', ascending=False).head(10)
fig = px.bar(pm_reaccion, x='TR', y='PMP', color='PMP',
             title='Grafico 6: Efecto del Mecanismo en el PM del Producto')
fig.show()


In [187]:
# GRAFICO 7
def tox(s):
    s = str(s).lower()
    if 'benceno' in s or 'tolueno' in s or 'cloroformo' in s:
        return 'Alta'
    elif 'agua' in s or 'etanol' in s:
        return 'Baja'
    return 'Media'
df['Tox'] = df['S1'].apply(tox)
tox_data = df.groupby(['NR', 'Tox']).size().reset_index(name='count')
fig = px.bar(tox_data, x='NR', y='count', color='Tox',
             title='Grafico 7: Criticidad de Solventes por Reaccion')
fig.show()

In [188]:
# GRAFICO 8
def tipo(s):
    s = str(s).lower()
    if 'agua' in s or 'etanol' in s:
        return 'Verde'
    elif 'benceno' in s or 'tolueno' in s:
        return 'Toxico'
    return 'Moderado'
df['TipoSol'] = df['S1'].apply(tipo)
tabla = pd.crosstab(df['TR'], df['TipoSol'], normalize='index') * 100
tabla = tabla.reset_index()
fig = px.bar(tabla, x='TR', y=['Verde', 'Moderado', 'Toxico'],
             title='Grafico 8: Clasificacion Ecologica de Solventes')
fig.show()

In [189]:
# GRAFICO 9
pka_df = df.dropna(subset=['pKa S', 'pKa P', 'Grupo'])
fig = px.scatter(pka_df, x='pKa S', y='pKa P', color='Grupo',
                 title='Grafico 9: Cambio de Acidez en la Reaccion')
fig.add_shape(type='line', x0=0, y0=0, x1=50, y1=50, line=dict(dash='dash', color='gray'))
fig.show()

In [190]:
# GRAFICO 10
cat = df['C1'].dropna().value_counts().reset_index().head(10)
cat.columns = ['Catalizador', 'Frecuencia']
cat['Acum'] = cat['Frecuencia'].cumsum()
cat['Pct'] = 100 * cat['Acum'] / cat['Frecuencia'].sum()
fig = go.Figure()
fig.add_trace(go.Bar(x=cat['Catalizador'], y=cat['Frecuencia'], name='Frecuencia'))
fig.add_trace(go.Scatter(x=cat['Catalizador'], y=cat['Pct'], name='% Acumulado', yaxis='y2'))
fig.update_layout(title='Grafico 10: Catalizadores mas Utilizados (Pareto)',
                  xaxis_title='Catalizador', yaxis_title='Frecuencia',
                  yaxis2=dict(title='% Acumulado', overlaying='y', side='right'))
fig.show()

In [191]:
# GRAFICO 11
comp = df.groupby('Grupo')[['LD5 S', 'LD5 P']].mean().reset_index()
comp_melt = comp.melt(id_vars='Grupo', var_name='Tipo', value_name='LD50')
fig = px.bar(comp_melt, x='Grupo', y='LD50', color='Tipo', barmode='group',
             title='Grafico 11: Comparacion de Toxicidad (Sustrato vs Producto)')
fig.show()